# 04 — Network Reconnaissance Automation

## What This Is
Reconnaissance is the intelligence-gathering phase of a security assessment. Automated recon tools (Nmap, Shodan, Amass) correlate data from DNS, WHOIS, certificate transparency, and port scans to map attack surfaces.

## Why It Matters
A thorough asset inventory is the foundation of both offensive (finding attack entry points) and defensive (knowing what to protect) security. Security teams that don't map their own surface have no basis for risk prioritisation.

## Where the Community Stands
ASM (Attack Surface Management) platforms have productized continuous recon. Bug bounty hunters rely on automated recon pipelines. Responsible disclosure programmes require scoping — only test systems explicitly listed in scope.

## Authorized Testing Context
All recon simulation here operates on synthetic data. Never run port scans or DNS enumeration against targets without explicit written permission from the asset owner.

In [ ]:
import random
import ipaddress
import hashlib
import re
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from datetime import datetime, timedelta

# --- Synthetic asset database (replaces live scans) ---
SYNTHETIC_HOSTS = [
    {'ip': '10.0.1.1',   'hostname': 'gateway.corp.local',      'ports': [22, 80, 443, 8080]},
    {'ip': '10.0.1.10',  'hostname': 'webserver.corp.local',    'ports': [80, 443, 8443]},
    {'ip': '10.0.1.20',  'hostname': 'db.corp.local',           'ports': [3306, 5432]},
    {'ip': '10.0.1.30',  'hostname': 'ldap.corp.local',         'ports': [389, 636]},
    {'ip': '10.0.1.40',  'hostname': 'mail.corp.local',         'ports': [25, 587, 993]},
    {'ip': '10.0.1.50',  'hostname': 'jenkins.corp.local',      'ports': [8080, 50000]},
    {'ip': '10.0.1.60',  'hostname': 'k8s-api.corp.local',      'ports': [6443]},
    {'ip': '10.0.1.70',  'hostname': 'elasticsearch.corp.local','ports': [9200, 9300]},
]

SERVICE_MAP = {
    22:    'SSH',     80:    'HTTP',    443:   'HTTPS',
    3306:  'MySQL',   5432:  'PostgreSQL', 389: 'LDAP',
    636:   'LDAPS',   25:    'SMTP',    587:   'SMTP/TLS',
    993:   'IMAPS',   8080:  'HTTP-alt',8443:  'HTTPS-alt',
    6443:  'K8s-API', 9200:  'ES-HTTP', 9300:  'ES-Transport',
    50000: 'Jenkins-JNLP',
}

print(f'Synthetic network: {len(SYNTHETIC_HOSTS)} hosts')

In [ ]:
@dataclass
class HostRecord:
    ip: str
    hostname: str
    open_ports: List[int]
    services: Dict[int, str] = field(default_factory=dict)
    banners: Dict[int, str] = field(default_factory=dict)
    risk_score: float = 0.0

class ReconEngine:
    """Simulates multi-phase reconnaissance pipeline."""

    # High-risk port categories
    CRITICAL_SERVICES = {6443, 9200, 50000}  # K8s, Elasticsearch, Jenkins
    DB_PORTS          = {3306, 5432, 27017, 6379, 5984}
    AUTH_PORTS        = {389, 636, 88, 464}   # LDAP, Kerberos

    def __init__(self, subnet: str = '10.0.1.0/24'):
        self.subnet  = ipaddress.ip_network(subnet, strict=False)
        self.results: List[HostRecord] = []

    def _simulate_banner(self, port: int) -> str:
        banners = {
            22:    'SSH-2.0-OpenSSH_8.2p1 Ubuntu-4ubuntu0.5',
            80:    'nginx/1.18.0',
            443:   'nginx/1.18.0 (TLS: TLSv1.2)',
            8080:  'Apache Tomcat/9.0.54',
            9200:  'Elasticsearch 7.10.0',
            6443:  'Kubernetes API Server v1.24.0',
            50000: 'Jenkins 2.387.1',
            3306:  'MySQL 8.0.31',
        }
        return banners.get(port, f'service/{SERVICE_MAP.get(port, "unknown")}')

    def port_scan(self, host_data: dict) -> HostRecord:
        rec = HostRecord(
            ip=host_data['ip'],
            hostname=host_data['hostname'],
            open_ports=host_data['ports'],
        )
        for port in rec.open_ports:
            rec.services[port] = SERVICE_MAP.get(port, 'unknown')
            rec.banners[port]  = self._simulate_banner(port)
        return rec

    def score_host(self, rec: HostRecord) -> float:
        score = 0.0
        port_set = set(rec.open_ports)
        if port_set & self.CRITICAL_SERVICES:
            score += 4.0
        if port_set & self.DB_PORTS:
            score += 3.0
        if port_set & self.AUTH_PORTS:
            score += 2.5
        if 22 in port_set:
            score += 1.0
        score += len(rec.open_ports) * 0.2
        return round(score, 2)

    def run(self) -> List[HostRecord]:
        for host_data in SYNTHETIC_HOSTS:
            rec = self.port_scan(host_data)
            rec.risk_score = self.score_host(rec)
            self.results.append(rec)
        self.results.sort(key=lambda r: r.risk_score, reverse=True)
        return self.results

engine = ReconEngine()
hosts  = engine.run()
print(f'Scanned {len(hosts)} hosts\n')
print(f'  {"IP":<16} {"Hostname":<32} {"Ports":<20} {"Risk"}')
print(f'  {"-"*16} {"-"*32} {"-"*20} {"-"*6}')
for h in hosts:
    ports_str = ','.join(str(p) for p in h.open_ports[:4])
    print(f'  {h.ip:<16} {h.hostname:<32} {ports_str:<20} {h.risk_score}')

## DNS Enumeration and Certificate Transparency
DNS subdomain enumeration and certificate transparency logs (crt.sh) reveal assets not otherwise visible. CertSpotter and Censys index all publicly-trusted TLS certificates, exposing internal hostnames.

In [ ]:
SYNTHETIC_SUBDOMAINS = [
    'www', 'mail', 'ftp', 'vpn', 'remote', 'dev', 'staging',
    'api', 'admin', 'jenkins', 'jira', 'confluence', 'gitlab',
    'grafana', 'kibana', 'elastic', 'mongo', 'redis', 'kafka',
]

def simulate_dns_enum(domain: str, wordlist: List[str]) -> List[Dict]:
    """Simulate DNS brute-force (no real DNS queries)."""
    results = []
    # Simulate ~40% hit rate
    rng = random.Random(hashlib.md5(domain.encode()).hexdigest())
    for sub in wordlist:
        fqdn = f'{sub}.{domain}'
        if rng.random() < 0.4:
            ip = f'10.{rng.randint(0,5)}.{rng.randint(0,3)}.{rng.randint(1,254)}'
            results.append({'fqdn': fqdn, 'ip': ip, 'source': 'dns_brute'})
    return results

def simulate_cert_transparency(domain: str) -> List[Dict]:
    """Simulate crt.sh certificate log results."""
    rng = random.Random(domain)
    names = ['*.'+domain, 'api.'+domain, 'internal.'+domain,
             'staging.'+domain, 'dev-api.'+domain]
    return [
        {'cn': n, 'issuer': 'Let\'s Encrypt', 'not_after': '2025-12-31'}
        for n in names if rng.random() < 0.7
    ]

domain = 'example.com'
dns_hits  = simulate_dns_enum(domain, SYNTHETIC_SUBDOMAINS)
cert_hits = simulate_cert_transparency(domain)

print(f'DNS enumeration ({len(SYNTHETIC_SUBDOMAINS)} words -> {len(dns_hits)} hits):')
for r in dns_hits[:5]:
    print(f"  {r['fqdn']:<35} {r['ip']}")
print(f'\nCert transparency ({len(cert_hits)} certs):')
for c in cert_hits:
    print(f"  {c['cn']}")

## Attack Surface Summary and Prioritisation
Good recon ends with a prioritised list of targets ranked by exposure. High-value targets: admin panels, databases exposed without authentication, developer tools (Jenkins, Grafana) left on default credentials.

In [ ]:
def generate_attack_surface_report(hosts: List[HostRecord], dns_hits: List[Dict]) -> None:
    print('=== Attack Surface Report ===')
    print(f'Total hosts discovered: {len(hosts)}')
    print(f'Total subdomains:       {len(dns_hits)}')
    print()

    critical = [h for h in hosts if h.risk_score >= 4.0]
    high     = [h for h in hosts if 2.0 <= h.risk_score < 4.0]
    medium   = [h for h in hosts if h.risk_score < 2.0]

    print(f'CRITICAL ({len(critical)} hosts):')
    for h in critical:
        svcs = ', '.join(h.services[p] for p in h.open_ports)
        print(f'  [{h.risk_score}] {h.hostname} — {svcs}')

    print(f'\nHIGH ({len(high)} hosts):')
    for h in high:
        svcs = ', '.join(h.services[p] for p in h.open_ports)
        print(f'  [{h.risk_score}] {h.hostname} — {svcs}')

    print(f'\nMEDIUM/LOW ({len(medium)} hosts):')
    for h in medium:
        print(f'  [{h.risk_score}] {h.hostname}')

    # Identify developer tools (common default-cred targets)
    dev_tools = [h for h in hosts if any(p in {8080,50000,9200} for p in h.open_ports)]
    if dev_tools:
        print(f'\nDeveloper tools exposed ({len(dev_tools)}): check for default credentials!')
        for h in dev_tools:
            print(f'  {h.hostname}')

generate_attack_surface_report(hosts, dns_hits)